In [2]:
import time

import faiss
import numpy as np

from sklearn.cluster import KMeans

In [8]:
DIM = 128
SIZES = [10_000, 100_000, 1_000_000] # might crash with 10_000_000 on a personal laptop due to memory constraints

In [12]:
def profile(name, sizes, d, build_index, search_index, k=10):
    results = []

    print(f"\n{name}")

    for n in sizes:
        database = np.random.randn(n, d).astype(np.float32)
        query = np.random.randn(d).astype(np.float32)

        start = time.time()
        index = build_index(database)
        index_elapsed = time.time() - start

        start = time.time()
        indices = search_index(index, query, k)
        search_elapsed = time.time() - start

        memory_gb = (n * d * 4) / (1024**3)

        result = {
            "name": name,
            "n": n,
            "d": d,
            "k": k,
            "index_seconds": index_elapsed,
            "search_seconds": search_elapsed,
            "memory_gb": memory_gb,
            "indices": indices,
        }

        results.append(result)

        print(
            f"n={n:>10,}: "
            f"index={index_elapsed*1000:>8.1f}ms, "
            f"search={search_elapsed*1000:>8.1f}ms, "
            f"memory={memory_gb:.2f}GB"
        )

    return results

# Brute Force

In [13]:
def brute_force_build_index(database):
    """No preparation needed; return the database as the searchable object."""
    return database


def brute_force_search_index(database, query, k=10):
    """Find k nearest neighbors by computing all L2 distances."""
    distances = np.sum((database - query) ** 2, axis=1)
    return np.argsort(distances)[:k]

profile(
    name="Brute force",
    sizes=SIZES,
    d=DIM,
    build_index=brute_force_build_index,
    search_index=brute_force_search_index,
)


Brute force
n=    10,000: index=     0.0ms, search=     1.5ms, memory=0.00GB
n=   100,000: index=     0.0ms, search=    11.2ms, memory=0.05GB
n= 1,000,000: index=     0.0ms, search=   193.6ms, memory=0.48GB


[{'name': 'Brute force',
  'n': 10000,
  'd': 128,
  'k': 10,
  'index_seconds': 9.5367431640625e-07,
  'search_seconds': 0.001458883285522461,
  'memory_gb': 0.00476837158203125,
  'indices': array([6503, 9718,  670, 1952, 9894, 2089, 4357, 2962, 5895, 7179])},
 {'name': 'Brute force',
  'n': 100000,
  'd': 128,
  'k': 10,
  'index_seconds': 2.1457672119140625e-06,
  'search_seconds': 0.011215925216674805,
  'memory_gb': 0.0476837158203125,
  'indices': array([64038, 42266, 56638, 66846, 89585, 65386, 14667, 48875, 75062,
         78443])},
 {'name': 'Brute force',
  'n': 1000000,
  'd': 128,
  'k': 10,
  'index_seconds': 2.1457672119140625e-06,
  'search_seconds': 0.19361209869384766,
  'memory_gb': 0.476837158203125,
  'indices': array([895521, 215191, 159093, 868460,  76528, 779579, 599654, 734030,
         518616, 431161])}]

# IVF: InVerted File index

Pros:
  - Faster search than brute force because it searches only selected clusters/lists.

  Cons:
  - Longer build/index time than brute force because it must train centroids and assign vectors.
  - Approximate search: it can miss true nearest neighbors if their clusters are not probed.

Similar:
  - Memory usage is roughly the same as brute force for IVF-Flat,  with some extra overhead for centroids, ids, and inverted-list structure.

## Educational example using K-Means clustering

In [14]:
def kmeans_ivf_build_index(database, nlist=1024, train_size=100_000):
    """
    Build an IVF-style index using k-means.

    nlist = number of clusters / inverted lists.
    """
    training_vectors = database[: min(train_size, len(database))]

    kmeans = KMeans(
        n_clusters=nlist,
        random_state=42,
        n_init=1,
    )
    kmeans.fit(training_vectors)

    centroids = kmeans.cluster_centers_.astype(np.float32)

    # Assign every database vector to its nearest centroid.
    labels = kmeans.predict(database)

    inverted_lists = []

    for list_id in range(nlist):
        indices = np.where(labels == list_id)[0]
        vectors = database[indices]

        inverted_lists.append(
            {
                "indices": indices,
                "vectors": vectors,
            }
        )

    return {
        "centroids": centroids,
        "inverted_lists": inverted_lists,
    }


def kmeans_ivf_search_index(index, query, k=10, nprobe=8):
    """
    Search an IVF-style index.

    nprobe = number of centroid lists to scan.
    """
    centroids = index["centroids"]
    inverted_lists = index["inverted_lists"]

    centroid_distances = np.sum((centroids - query) ** 2, axis=1)
    probe_list_ids = np.argsort(centroid_distances)[:nprobe]

    candidate_indices = []
    candidate_vectors = []

    for list_id in probe_list_ids:
        entry = inverted_lists[list_id]

        if len(entry["indices"]) == 0:
            continue

        candidate_indices.append(entry["indices"])
        candidate_vectors.append(entry["vectors"])

    if not candidate_vectors:
        return np.array([], dtype=np.int64)

    candidate_indices = np.concatenate(candidate_indices)
    candidate_vectors = np.vstack(candidate_vectors)

    distances = np.sum((candidate_vectors - query) ** 2, axis=1)
    nearest = np.argsort(distances)[:k]

    return candidate_indices[nearest]



profile(
    name="KMeans IVF",
    sizes=SIZES,
    d=DIM,
    build_index=lambda database: kmeans_ivf_build_index(
        database,
        nlist=1024,
        train_size=100_000,
    ),
    search_index=lambda index, query, k: kmeans_ivf_search_index(
        index,
        query,
        k=k,
        nprobe=8,
    ),
)



KMeans IVF
n=    10,000: index=   826.9ms, search=     0.2ms, memory=0.00GB
n=   100,000: index=  8872.7ms, search=     0.2ms, memory=0.05GB
n= 1,000,000: index=  9696.6ms, search=     1.1ms, memory=0.48GB


[{'name': 'KMeans IVF',
  'n': 10000,
  'd': 128,
  'k': 10,
  'index_seconds': 0.826894998550415,
  'search_seconds': 0.00016498565673828125,
  'memory_gb': 0.00476837158203125,
  'indices': array([4566, 2729, 3565, 1057, 6862, 4722, 1874, 4618, 3587, 3383])},
 {'name': 'KMeans IVF',
  'n': 100000,
  'd': 128,
  'k': 10,
  'index_seconds': 8.872670888900757,
  'search_seconds': 0.00021791458129882812,
  'memory_gb': 0.0476837158203125,
  'indices': array([49912, 60407, 80042, 61720, 10082,  7967, 93379, 55172, 79723,
         12798])},
 {'name': 'KMeans IVF',
  'n': 1000000,
  'd': 128,
  'k': 10,
  'index_seconds': 9.696568965911865,
  'search_seconds': 0.0010530948638916016,
  'memory_gb': 0.476837158203125,
  'indices': array([866091, 277212,  53966, 105899,  67234, 250112, 539993, 383394,
         854158, 481067])}]

## Optimised FAISS IVF implementation

In [15]:
def faiss_ivf_build_index(database, d, nlist=1024, train_size=100_000):
    """Build and populate a FAISS IVF index."""
    quantizer = faiss.IndexFlatL2(d)
    index = faiss.IndexIVFFlat(quantizer, d, nlist)

    training_data = database[: min(train_size, len(database))]
    index.train(training_data)
    index.add(database)

    return index


def faiss_ivf_search_index(index, query, k=10):
    """Search a FAISS index."""
    distances, indices = index.search(query.reshape(1, -1), k)
    return indices[0]


profile(
    name="FAISS IVF",
    sizes=SIZES,
    d=DIM,
    build_index=lambda database: faiss_ivf_build_index(database, d=DIM),
    search_index=faiss_ivf_search_index,
)


FAISS IVF
n=    10,000: index=    81.6ms, search=     0.7ms, memory=0.00GB


WARNING clustering 10000 points to 1024 centroids: please provide at least 39936 training points


n=   100,000: index=   684.2ms, search=     0.4ms, memory=0.05GB
n= 1,000,000: index=  1322.8ms, search=     0.1ms, memory=0.48GB


[{'name': 'FAISS IVF',
  'n': 10000,
  'd': 128,
  'k': 10,
  'index_seconds': 0.08164095878601074,
  'search_seconds': 0.0006582736968994141,
  'memory_gb': 0.00476837158203125,
  'indices': array([ 295,  708, 8219, 7849, 5898,  434, 6670, 6681, 6503, 1913])},
 {'name': 'FAISS IVF',
  'n': 100000,
  'd': 128,
  'k': 10,
  'index_seconds': 0.6841700077056885,
  'search_seconds': 0.0003790855407714844,
  'memory_gb': 0.0476837158203125,
  'indices': array([88429,   953, 34420, 86608, 37176, 62243, 69431,  6755, 18957,
         37770])},
 {'name': 'FAISS IVF',
  'n': 1000000,
  'd': 128,
  'k': 10,
  'index_seconds': 1.322796106338501,
  'search_seconds': 5.2928924560546875e-05,
  'memory_gb': 0.476837158203125,
  'indices': array([175627, 462996, 931497, 614867, 192254, 375139, 756423, 973675,
         593866, 575389])}]

# Product Quantization (PQ)

Pros:
- Much lower memory usage than brute force or IVF-Flat because vectors are stored as compact codes.
- Can make large-scale search practical when full vectors do not fit comfortably in memory.
- Search can be faster than brute force because distances are computed using small lookup tables.

Cons:
- Longer build/index time than brute force because it must train codebooks and encode vectors.
- Search is approximate: compression loses information, so recall can be lower.
- May be slower than IVF-Flat for some sizes/settings unless combined with an inverted index.

## Educational example using K-Means

In [16]:
def train_pq_codebooks(training_vectors, m=8, k=256):
    """
    Train Product Quantization codebooks.

    For each subspace, run k-means to find k representative centroids.
    These centroids form the "vocabulary" for encoding subvectors.
    """
    n, d = training_vectors.shape
    d_sub = d // m
    codebooks = []

    for j in range(m):
        # Extract subvectors for this subspace
        start, end = j * d_sub, (j + 1) * d_sub
        subvectors = training_vectors[:, start:end]

        # Run k-means
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=1)
        kmeans.fit(subvectors)

        codebooks.append(kmeans.cluster_centers_.astype(np.float32))
        # print(f"Codebook {j+1}/{m}: {k} centroids of dim {d_sub}")

    return codebooks


def encode_pq(vectors, codebooks):
    """Encode vectors as PQ codes (centroid indices)."""
    n, d = vectors.shape
    m = len(codebooks)
    d_sub = d // m
    k = codebooks[0].shape[0]

    # Use uint8 if k <= 256, else uint16
    dtype = np.uint8 if k <= 256 else np.uint16
    codes = np.zeros((n, m), dtype=dtype)

    for j, codebook in enumerate(codebooks):
        subvectors = vectors[:, j*d_sub:(j+1)*d_sub]

        # Find nearest centroid for each subvector
        distances = np.sum(
            (subvectors[:, np.newaxis] - codebook) ** 2, axis=2
        )
        codes[:, j] = np.argmin(distances, axis=1)

    return codes


def kmeans_pq_build_index(database, m=16, k=256, train_size=100_000):
    training_vectors = database[: min(train_size, len(database))]
    codebooks = train_pq_codebooks(
        training_vectors,
        m=m,
        k=k,
    )
    codes = encode_pq(database, codebooks)
    return {
        "codebooks": codebooks,
        "codes": codes,
    }


def kmeans_pq_search_index(index, query, k=10):
    codebooks = index["codebooks"]
    codes = index["codes"]

    d = query.shape[0]
    m = len(codebooks)
    d_sub = d // m

    lookup_tables = []

    for j, codebook in enumerate(codebooks):
        start, end = j * d_sub, (j + 1) * d_sub
        query_subvector = query[start:end]

        distances = np.sum((codebook - query_subvector) ** 2, axis=1)
        lookup_tables.append(distances)

    approximate_distances = np.zeros(codes.shape[0], dtype=np.float32)

    for j, table in enumerate(lookup_tables):
        approximate_distances += table[codes[:, j]]

    return np.argsort(approximate_distances)[:k]


profile(
    name="KMeans PQ",
    sizes=SIZES,
    d=DIM,
    build_index=lambda database: kmeans_pq_build_index(
        database,
        m=16,
        k=256,
        train_size=100_000,
    ),
    search_index=kmeans_pq_search_index,
)


KMeans PQ
n=    10,000: index=  2050.4ms, search=     2.8ms, memory=0.00GB
n=   100,000: index= 38185.2ms, search=     6.9ms, memory=0.05GB
n= 1,000,000: index=230831.3ms, search=    84.0ms, memory=0.48GB


[{'name': 'KMeans PQ',
  'n': 10000,
  'd': 128,
  'k': 10,
  'index_seconds': 2.050366163253784,
  'search_seconds': 0.0028290748596191406,
  'memory_gb': 0.00476837158203125,
  'indices': array([ 201, 2444, 4561, 2599, 5238, 7249,  108, 6854, 3255, 9490])},
 {'name': 'KMeans PQ',
  'n': 100000,
  'd': 128,
  'k': 10,
  'index_seconds': 38.18516397476196,
  'search_seconds': 0.006895780563354492,
  'memory_gb': 0.0476837158203125,
  'indices': array([63654, 55382, 82266, 42871, 41879, 72646, 69302, 18726, 48126,
         90429])},
 {'name': 'KMeans PQ',
  'n': 1000000,
  'd': 128,
  'k': 10,
  'index_seconds': 230.8312759399414,
  'search_seconds': 0.08404707908630371,
  'memory_gb': 0.476837158203125,
  'indices': array([ 58371, 549927, 371590,  58892, 467587, 610058, 959235, 771140,
         915260, 360376])}]

## Optimised FAISS PQ implementation

In [17]:
def faiss_pq_build_index(database, d, m=16, nbits=8, train_size=100_000):
    index = faiss.IndexPQ(d, m, nbits)

    training_data = database[: min(train_size, len(database))]
    index.train(training_data)
    index.add(database)

    return index


def faiss_pq_search_index(index, query, k=10):
    distances, indices = index.search(query.reshape(1, -1), k)
    return indices[0]


profile(
    name="FAISS PQ",
    sizes=SIZES,
    d=DIM,
    build_index=lambda database: faiss_pq_build_index(
        database,
        d=DIM,
        m=16,
        nbits=8,
        train_size=100_000,
    ),
    search_index=faiss_pq_search_index,
)


FAISS PQ
n=    10,000: index=   562.2ms, search=     1.5ms, memory=0.00GB
n=   100,000: index=   731.6ms, search=     0.6ms, memory=0.05GB
n= 1,000,000: index=  2430.4ms, search=     3.0ms, memory=0.48GB


[{'name': 'FAISS PQ',
  'n': 10000,
  'd': 128,
  'k': 10,
  'index_seconds': 0.5622048377990723,
  'search_seconds': 0.0014820098876953125,
  'memory_gb': 0.00476837158203125,
  'indices': array([8034, 2130, 8612, 3442, 4763, 3994, 2749,  714, 3409, 2867])},
 {'name': 'FAISS PQ',
  'n': 100000,
  'd': 128,
  'k': 10,
  'index_seconds': 0.731569766998291,
  'search_seconds': 0.0005979537963867188,
  'memory_gb': 0.0476837158203125,
  'indices': array([63616, 99574, 69427, 58620, 59517, 18688, 67326, 44402,  3005,
         57137])},
 {'name': 'FAISS PQ',
  'n': 1000000,
  'd': 128,
  'k': 10,
  'index_seconds': 2.4304118156433105,
  'search_seconds': 0.00298309326171875,
  'memory_gb': 0.476837158203125,
  'indices': array([556200, 467694, 247764, 494969, 231258, 800654, 439675, 334455,
         603243, 328352])}]